# Python Object-Oriented Programming

---

1. [Types of methods](#Methods-in-OOP),
    - [static method](#Static-method),
    - [class method](#Class-method),
2. [underscores in Python](#Underscore-functions-in-Python),
    - [last expression & skip expression](#Last-expression),
    - [private variables](#Private-variable-~weak-private),
    - [protected variables](#Protected-variable-~strong-private),
    - [keyword override](#Keyword-override),
    - [magic methods](#Magic-methods).
    
---

<img src="https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Ftse1.mm.bing.net%2Fth%3Fid%3DOIP.6MU4ScqH-1D93XRpyEYF4AHaHa%26pid%3DApi&f=1&ipt=03e1caf2482e1b90758340dcb676d181c012a20af163e9308f80f29ae4f6cc27&ipo=images" width="250" style="margin-left:auto; margin-right:auto"/>

## Methods in OOP

---
You can describe a **regular method** (also called an *instance method*) as a *user-defined function*..

In [ ]:
class Zamestnanec:
    
    def __init__(self, jmeno: str, prijmeni: str):
        self.jmeno = jmeno
        self.prijmeni = prijmeni
        
    def vytvor_cele_jmeno(self) -> str:
        return f"{self.jmeno} {self.prijmeni}"

<br>

...which belongs to the **class itself**.

So it is not possible to work with an *instance method* as with a standalone function:

In [ ]:
vytvor_cele_jmeno()  # BAD

In [ ]:
matous = Zamestnanec("Matouš", "Holinka")

In [ ]:
matous.vytvor_cele_jmeno()

<br>

A method that has, among its parameters, the special keyword `self`.

Thanks to that it can access both the class (and its *attributes*) and its *instances*:

In [ ]:
print(matous.jmeno, matous.prijmeni)

<br>

However, the way of writing methods may differ.

There are several variants of using methods:
1. **instance** method,
2. **static** method,
3. **class** method.

<br>

### Static method

---

However, not every method necessarily has to belong to a class:

In [ ]:
class GeneratorTextovehoSouboru:
    def __init__(self, jmeno_souboru: str):
        self.jmeno_souboru = jmeno_souboru
        self.obsah_souboru = list()

    def spoj_zadany_obsah(self, spojovac: str = '-', *obsah) -> str:
        return spojovac.join(obsah)

In such a case, it is unnecessary to use the `self` parameter, because it is not used anywhere in the `spoj_zadany_obsah` method.

Then it is good to ask yourself two simple questions:
1. **Is it necessary for it to be part of a class?**
2. **Where should I correctly place such a method?**

<br>

#### Placing the method outside the class

---

If it is an *object* that can be reused across **different classes** or **modules**, move the *method* outside the *class*.

It is more appropriate to turn it into a standalone *user-defined function*: 

In [ ]:
def spoj_zadany_obsah(*obsah, spojovac: str = '-') -> str:
    return spojovac.join(obsah)


class GeneratorTextovehoSouboru:
    def __init__(self, jmeno_souboru: str):
        self.jmeno_souboru = jmeno_souboru
        self.obsah_souboru = list()

In [ ]:
muj_txt_soubor = GeneratorTextovehoSouboru("muj_soubor.txt")

In [ ]:
obsah_1 = spoj_zadany_obsah("a", "b", "c")

In [ ]:
print(
    muj_txt_soubor.jmeno_souboru,
    obsah_1,
    sep='\n'
)

<br>

If you need to change the separator, define it at the end (as a *keyword argument*).

In [ ]:
obsah_2 = spoj_zadany_obsah("a", "b", "c", spojovac='#')

In [ ]:
print(obsah_2)

<br>

#### Placing the method in the class with a decorator

---

If you want such a method to be **part of the class** (or it logically belongs there), you can keep it **inside the class**.

In that case, it is a good idea to tell others, **using a decorator**, that it is a *static method*.

In [ ]:
from pathlib import Path

In [ ]:
class GeneratorTextovehoSouboru:

    def __init__(self, jmeno_souboru: str):
        self.obsah = list()
        self.jmeno_souboru = jmeno_souboru
            
    @staticmethod
    def existuje_soubor(jmeno_souboru: str) -> bool:
        return Path(jmeno_souboru).is_file()

<br>

You can immediately recognize a *static method* by:
1. The **decorator** `@staticmethod`,
2. The **missing** `self` parameter.

It is a method that has a logical connection (is encapsulated) to a **specific class** or **instance** (easier to find, test, and document).

In [ ]:
poznamky = GeneratorTextovehoSouboru("moje_poznamky.txt")

In [ ]:
poznamky.existuje_soubor("moje_poznamky.txt")

In this case your static method works perfectly.

There is **no** file `moje_poznamky.txt` in the current directory!

In [ ]:
!ls -l  # Unixový příkaz

<br>

But once you **create** it:

In [ ]:
!touch moje_poznamky.txt

In [ ]:
poznamky.existuje_soubor("moje_poznamky.txt")

Then the file will be found reliably.

You can also work without **creating an instance of the class**.

That is, just by using the class name itself where the *static method* is defined.

In [ ]:
GeneratorTextovehoSouboru.existuje_soubor("moje_poznamky.txt")

<br>

**Attention**, this time you cannot work with just the method name!

In [ ]:
existuje_soubor("moje_poznamky.txt")

...because the *static method* itself **does not** have access **neither to instance attributes, nor to class attributes**:

In [ ]:
class GeneratorTextovehoSouboru:

    def __init__(self, jmeno_souboru: str):
        self.obsah = list()
        self.jmeno_souboru = jmeno_souboru
           
    @staticmethod
    def existuje_soubor(jmeno_souboru: str) -> bool:
        return Path(self.jmeno_souboru).is_file()   # chybně umístěný parametr "self"

In [ ]:
poznamky_dalsi = GeneratorTextovehoSouboru("moje_poznamky.txt")

In [ ]:
poznamky_dalsi.existuje_soubor("moje_poznamky.txt")

<br>

#### Recap of static methods

---

When to use the `@staticmethod` decorator:

1. When the method you created **will not be used in other classes**,
2. when it is **logically tied to the purpose of the class**.

In [ ]:
class MatematickeOperace:
    """Object that groups various mathematical operations"""

    @staticmethod
    def secti_dve_cisla(x: int, y: int) -> int:
        return x + y
    
    @staticmethod
    def odecti_dve_cisla(x: int, y: int) -> int:
        return x - y

In [ ]:
soucet_1 = MatematickeOperace.secti_dve_cisla(4, 6)

In [ ]:
rozdil_1 = MatematickeOperace.odecti_dve_cisla(121, 31)

In [ ]:
print(soucet_1, rozdil_1, sep='\n')

<br>

### 🧠 EXERCISE 🧠, Create a class `GeneratorTextovehoSouboru` that checks the file name suffix:
---

1. Copy the class `GeneratorTextovehoSouboru`,
2. keep only the instance attribute `jmeno_souboru`,
3. write a **static method** `je_soubor_txt`,
4. check whether the parameter `jmeno` has the extension `".txt"` or not,
5. if it has the `txt` suffix, return `True`, otherwise return `False`.

In [ ]:
# Write your solution here
class GeneratorTextovehoSouboru:
    
    @staticmethod
    def je_soubor_txt(jmeno: str, pripona: str = '.txt') -> bool:  # True/False --> islower, isupper
        return Path(jmeno).suffix == pripona

In [ ]:
GeneratorTextovehoSouboru.je_soubor_txt("muj_soubor.txt")

In [ ]:
GeneratorTextovehoSouboru.je_soubor_txt("muj_soubor.csv")

<details>
    <summary>▶️ Řešení</summary>
    
```python
class GeneratorTextovehoSouboru:

    def __init__(self, jmeno_souboru: str):
        self.obsah = list()
        self.jmeno_souboru = jmeno_souboru

    @staticmethod
    def je_soubor_txt(jmeno_souboru: str, suffix: str = '.txt') -> bool:
        return Path(jmeno_souboru).suffix == suffix


GeneratorTextovehoSouboru.je_soubor_txt("soubor.txt")
```
</details>

<br>

### Class method

---

Sometimes you need to adjust the **standard behaviour** of your class.

In [ ]:
class Zamestnanec:
    def __init__(self, jmeno: str, prijmeni: str, mzda: int):
        self.jmeno = jmeno
        self.prijmeni = prijmeni
        self.mzda = mzda

<br>

Creating a new employee instance normally looks like this:

In [ ]:
matous = Zamestnanec('Matouš', 'Holinka', 80_000)
karolina = Zamestnanec('Karolína', 'Šikovná', 100_000)

In [ ]:
print(matous.mzda, karolina.mzda, sep='\n')

<br>

Sometimes it is not easy to connect to an existing system.

For example, if someone starts sending you data about new employees as a *table-like* or *text file*.

In [ ]:
petr = "Petr;Svetr;110_000"
jan = "Jan;Novák;140_000"

<br>

How can I feed such inputs into the existing `Zamestnanec` class?

This is exactly a scenario for using **another decorator**, this time `@classmethod`.

In [ ]:
class Zamestnanec:
    def __init__(self, jmeno: str, prijmeni: str, mzda: int):
        self.jmeno = jmeno
        self.prijmeni = prijmeni
        self.mzda = mzda
        
    @classmethod
    def from_separated_string(cls, zadany_str: str, oddelovac: str = ';'):
        """
        Create a new instance of the `Zamestnanec` class from the given string.
        """
        try:
            jmeno, prijmeni, mzda = zadany_str.split(oddelovac)
            
        except Exception:
            # logging.WARNING()
            instance = None
        else:
            instance = cls(jmeno, prijmeni, mzda)
        finally:
            return instance

<br>

A **class method** is easy to recognise:
1. The **decorator** `@classmethod`,
2. the **missing** helper parameter `self`,
3. a new helper parameter **`cls`** appears.

Instead of `cls` you can technically use any name.

But again, it is a convention so that other developers can easily see that this is a *class method*.

In [ ]:
petr = "Petr;Svetr;110_000"
jan  = "Jan;Novák;140_000"

In [ ]:
petr_i = Zamestnanec.from_separated_string(petr)

In [ ]:
jan_i = Zamestnanec.from_separated_string(jan)

In [ ]:
print(
    type(petr_i.mzda),
    jan_i.mzda,
    sep='\n'
)

The real benefit of the `@classmethod` decorator is that it lets you **create an alternative class constructor**.

<br>

Otherwise, you would have to **rewrite and split the values** manually, which is not very practical:

In [ ]:
lukas = "Lukáš;Nový;90_000"

In [ ]:
lukas = Zamestnanec(*lukas.split(";"))

In [ ]:
print(
    lukas.jmeno,
    lukas.prijmeni,
    lukas.mzda,
    sep="\n"
)

In [ ]:
class Zamestnanec:
    zvyseni_mzdy = 1.06

    def __init__(self, jmeno: str, prijmeni: str, mzda: int):
        self.jmeno = jmeno
        self.prijmeni = prijmeni
        self.mzda = mzda

    @classmethod
    def zvys_mzdu(cls, hodnota: int):
        """
        Overwrite the value of the class attribute `zvyseni_mzdy`.
        """
        cls.zvyseni_mzdy = hodnota

In [ ]:
matous = Zamestnanec('Matouš', 'Holinka', 80_000)
karolina = Zamestnanec('Karolína', 'Šikovná', 100_000)

In [ ]:
print(
    matous.zvyseni_mzdy,
    karolina.zvyseni_mzdy,
    sep="\n"
)

In [ ]:
Zamestnanec.zvys_mzdu(1.11)

In [ ]:
print(
    matous.zvyseni_mzdy,
    karolina.zvyseni_mzdy,
    sep="\n"
)

You will most often see the `@classmethod` decorator on methods named something like:
- `from_`,
- `make_`,
- `create_`.

Their goal is simply to **make it easier to construct new instances** (from various files and inputs in general).

[Link, pandas library](https://github.com/pandas-dev/pandas/blob/55d78caa38925e8cf94623adcd0721cc15a56bdd/pandas/core/indexes/range.py#L171)

[Link, datetime module](https://github.com/python/cpython/blob/main/Lib/_pydatetime.py#L983)

### Summary of methods

---

1. **instance method** – can modify **both instance objects and class objects** (it sees both the class and the instance),
2. **static method (`@staticmethod`)** – cannot modify **either instance objects or class objects** (it sees neither class nor instance),
3. **class method (`@classmethod`)** – can modify class objects, but cannot modify instance objects (it sees the class, but not the instance).


<br>

### 🧠 EXERCISE 🧠, Create a class `Produkt` with the following requirements:
---

1. Define a class `Produkt` with the class attribute `pocet_produktu`,
2. in the instance constructor, pass the parameters `nazev`, `cena` and `skladem` in this order,
3. for every instance, increment `pocet_produktu` by 1,
4. create a method `naskladni_produkt` that takes a single parameter `mnozstvi`,
5. create a method `prodej` that checks whether the amount `mnozstvi` can be sold,
6. create a static method `vypocet_ceny_vc_dane` that adds 21% tax to the price,
7. create a class method `from_json` that reads a JSON file and prepares an instance automatically.

In [ ]:
class Produkt:
    pocet_produktu = 0
    
    def __init__(self, nazev, cena, skladem):
        self.nazev = nazev
        self.cena = cena
        self.skladem = skladem
        Produkt.pocet_produktu += 1

    def prodej(self, mnozstvi):
        if mnozstvi >= self.skladem:
            return True
        else:
            return False
    
    def vypocet_ceny_vc_dane(self, daň = 1.21):
        return self.cena * daň

    @classmethod
    def naskladni_produkt(cls, mnozstvi):
         cls.pocet_produktu += mnozstvi
    
    @classmethod
    def from_json(soubor):
        pass

In [ ]:
produkt1 = Produkt("Notebook", 20000, 10)
produkt2 = Produkt("Telefon", 15000, 15)
produkt3 = Produkt("Sluchátka", 2000, 50)

<details>
    <summary>▶️ Solution</summary>
    
```python
class Produkt:
    pocet_produktu: int = 0

    def __init__(self, nazev: str, cena: str, skladem: int = 0):
        self.nazev = nazev
        self.cena = cena
        self.skladem = skladem
        Produkt.pocet_produktu += 1

    def naskladni(self, mnozstvi: int) -> None:
        self.skladem += mnozstvi

    def prodej(self, mnozstvi: int) -> None:
        if mnozstvi <= self.skladem:
            self.skladem -= mnozstvi
        else:
            print(f"There is not enough of product {self.name} in stock.")

    @classmethod
    def from_json(cls, jmeno_souboru: str) -> dict:
        try:
            obsah_json = read_json(jmeno_souboru, orient="index")

        except Exception:
            content_dict = {}
        else:
            content = obsah_json.to_dict()
            content_dict = cls(
                content[0]["nazev"],
                content[0]["cena"],
                content[0]["mnozstvi"]
            )
        finally:
            return content_dict

    @staticmethod
    def vypocet_cenu_vc_dane(cena: float) -> float:
        """
        Return the product price including 21% tax.

        :param cena: price without tax,
        :type cena: float
        :return: price including tax,
        :rtype: float
        """
        return cena * 1.21
```
</details>

<img src="https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Ftse1.mm.bing.net%2Fth%3Fid%3DOIP.W1bO-UE-mgx7us8eNrJTsQHaHa%26pid%3DApi&f=1&ipt=eaddf3d0ad49228dbd8d3a969c2c025036cee17f5c70daad917debf57a2f743e&ipo=images" width="250" style="margin-left:auto; margin-right:auto"/>

## Underscore functions in Python

---

The underscore is a **syntactic character** in Python that has several important meanings:
1. last expression `_`,
2. ignore an expression `_`,
3. private variable `_name`,
4. protected variable `__name`,
5. overridden keyword `class_`,
6. magic method `__init__`.

<br>

### Last expression

---

The first (although not very practical) use is to access the value of the last expression:

In [ ]:
"Marek"

In [ ]:
_

In [ ]:
_.upper()

In [ ]:
1 + 119

In [ ]:
_

In [ ]:
_ + 30

<br>

You will appreciate this behaviour mainly in *notebook* and *interpreter* environments.

In a source file, it is almost never used this way.


### Ignore an expression

---

The underscore becomes really useful in scripts as an **indicator of unused values**:

In [ ]:
import time

In [ ]:
for predmet in range(5):  # BAD
    print("Checking status..")
    time.sleep(1)

In [ ]:
for _ in range(5):     # EXCELLENT
    print("Checking status..")
    time.sleep(1)

<br>

Giving a name to the loop variable in the example above makes no sense.

You **do not need to use it** anywhere in the loop body.

It is therefore a good idea to signal this to other readers of the code.

<br>

Another use case is *iterating* over multiple values when you only need **one specific value**:

In [ ]:
for index, jmeno in enumerate(("Matouš", "Marcel", "Filip")):  # BAD
    print(index)

In [ ]:
for index, _ in enumerate(("Matouš", "Marcel", "Filip")):      # EXCELLENT
    print(index)

<br>

Again, other developers can now immediately see that the essential work in the loop body is done with the index.

<br>

Another common pattern is **multiple assignment**:

In [ ]:
jmeno, prijmeni, mzda = "Filip;Svědomitý;90_000".split(";")  # BAD

<br>

Again, there is nothing fundamentally wrong with this code.

But if you only need to unpack the value into the variable `jmeno`:

In [ ]:
jmeno, _, _ = "Filip;Svědomitý;90_000".split(";")            # GOOD

In [ ]:
print(jmeno)

<br>

Alternatively, you can pack the remaining unused values into a `list`:

In [ ]:
jmeno, *_ = "Filip;Svědomitý;90_000".split(";")              # EXCELLENT

In [ ]:
print(jmeno)

In [ ]:
print(_)

<br>

### Protected variable (~weak private)

---

You may know the terms **protected** and **public variable** from other languages.

Some languages support truly **internal** (*private*) variables (e.g. **Java**, **C#**):

In [ ]:
class VerifikatorLogu:
    def __init__(self, jmeno_souboru: str):
        self.jmeno_souboru = jmeno_souboru
        self.aktivni = True

    def oznam_stav(self):
        if self.aktivni:
            print(f"Checking log file: {self.jmeno_souboru}")

In [ ]:
verifikator_1 = VerifikatorLogu("muj_soubor.log")

In [ ]:
verifikator_1.oznam_stav()

<br>

If you add a single underscore as a *prefix* to the object name, you mark it as a *protected object*.

Python does not enforce protected variables like some other languages.

It treats this convention only **as a hint** or an indicator.

In [ ]:
class VerifikatorLogu:
    def __init__(self, jmeno_souboru: str):
        self.jmeno_souboru = jmeno_souboru
        self._aktivni = True  # WEAK PRIVATE

    def oznam_stav(self):
        if self._aktivni:
            print(f"Checking log file: {self.jmeno_souboru}")
        else:
            raise Exception("Uh-oh, some unexpected behaviour")

In [ ]:
verifikator_2 = VerifikatorLogu("muj_soubor.log")

In [ ]:
verifikator_2.oznam_stav()

In [ ]:
print(verifikator_2._aktivni)

In [ ]:
print(verifikator_2.aktivni)

<br>

The single underscore here works **only as an indicator**.

It tells other readers of the source code that this is a **protected variable**.

In other words, it is an **internal object** – *DO NOT MODIFY* it.

You may use it, but only within **one class (and its subclasses) or one module**.

<br>

It is usually a good idea to document what such an attribute is used for and how to work with it from the outside.

However, Python is very permissive and the *interpreter* will allow you to **use and overwrite** the variable anyway.

In [ ]:
verifikator_2._aktivni = False  # BAD

In [ ]:
verifikator_2.oznam_stav()

<br>

If you overwrite such a value, you may change the behaviour of the script, which is not desirable.

In the example above, a shorter time for checking the log might not be sufficient.

<br>

### Private variable (~strong private)

---

Using **two underscores** you can define **private** variables.

Such a variable (object) should be used **only within the class**:

In [ ]:
from time import sleep

In [ ]:
class VerifikatorLogu:
    """Object representing a log file."""
    
    def __init__(self, jmeno: str):
        self.jmeno = jmeno
        self._limit = 3                           # WEAK PRIVATE
        self.oznameni = self.__nastav_oznameni()  # STRONG PRIVATE
        
    def kontroluj_log(self) -> None:
        print(self.oznameni)

        for _ in range(self._limit):
            print(f"Checking log file..")
            sleep(1)
        else:
            print(f"File: '{self.jmeno}' is OK.")
            
    def __nastav_oznameni(self):
        return "Default notification"

<br>

This setup further highlights the **importance of the variable**.

It makes it harder for the user to simply overwrite or re-type the expression (to access it directly):

In [ ]:
verifikator_3 = VerifikatorLogu("protokol_1.log")

In [ ]:
print(verifikator_3._limit)

In [ ]:
verifikator_3.kontroluj_log()

<br>

This way the user will not find it so easily.

In the background, Python performs so-called *name mangling* (something like **name scrambling**).

It is still possible to look up the attribute manually and put in the effort to overwrite it:

In [ ]:
print(verifikator_3.oznameni)

<br>

The magic attribute `__dict__` allows you to list all instance attributes, where you will also find protected variables:

In [ ]:
class VerifikatorLogu:
    """Object representing a log file."""
    
    def __init__(self, jmeno: str):
        self.jmeno = jmeno
        self.__limit = 3  # STRONG PRIVATE
        
    def kontroluj_log(self) -> None:
        for _ in range(self.__limit):
            print(f"Checking log file..")
            sleep(1)
        else:
            print(f"File: '{self.jmeno}' is OK.")

In [ ]:
verifikator_4 = VerifikatorLogu("protokol_2.log")

In [ ]:
print(verifikator_4.__limit)

In [ ]:
print(verifikator_4.__dict__)

In [ ]:
verifikator_4._VerifikatorLogu__limit = 10

In [ ]:
print(verifikator_4._VerifikatorLogu__limit)

<br>

The *interpreter* protects such an attribute by **implicitly renaming** it.

#### 🧠 EXERCISE 🧠, Create a class `Zamestnanec` that contains:

---

1. The constructor `__init__` sets only the instance attribute `jmeno`,
2. `__init__` also creates a *weak private* attribute `vek` and a *strong private* attribute `plat`, both set to `None`,
3. write a method `nastav_vek` that takes the parameter `vek: int`,
4. the method `nastav_vek` sets the *weak private* attribute `vek` if the argument is 18 < x < 65 (otherwise it raises `Exception`),
5. write a method `nastav_plat` that takes the parameter `plat: int`,
6. the method `nastav_plat` sets the *strong private* attribute `plat` if the argument is x > 34_000 (otherwise it raises `Exception`),
7. create an instance method `zobraz_info` that prints: `"Jméno: <JMENO>, Věk: <VEK>"`,
8. create a *weak private* instance method `vypocitej_dan` that calculates 20% of the salary.

In [ ]:
class Zamestnanec:
    
    def __init__(self, jmeno):
        self.jmeno = jmeno
        self._vek = None
        self.__plat = None
    
    def nastav_vek(self, vek: int):
        if 18 < vek < 65:
            self._vek = vek
        else:
            raise Exception("Age does not meet the requirements!")
    
    def nastav_plat(self, plat: int):
        if plat > 34_000:
            self.__plat = plat
        else: 
            raise Exception("Salary not set because it is lower than 34 000!")
        
    def zobraz_info(self):
        print(f"{self.jmeno}, {self._vek} years old")
    
    def _vypocitej_dan(self, dan = 0.2):
        return self.__plat * dan

In [ ]:
matous = Zamestnanec("Matouš")

In [ ]:
print(matous._vek)

In [ ]:
matous.nastav_vek(50)

In [ ]:
print(matous._vek)

In [ ]:
matous.nastav_plat(50_000)

In [ ]:
matous.nastav_plat(20_000)

In [ ]:
matous.zobraz_info()

In [ ]:
matous._vypocitej_dan()

<details>
    <summary>▶️ Solution</summary>
    
```python
class Zamestnanec:
    
    def __init__(self, jmeno):
        self.jmeno = jmeno
        self._vek = None
        self.__plat = None

    def nastav_vek(self, vek: int) -> None:
        if 18 <= vek <= 65:
            self._vek = vek
        else:
            raise Exception(
                "The value of parameter 'vek' must not be less than 18 or greater than 65."
            )
    
    def nastav_plat(self, plat: int) -> None:
        if plat >= 34_000:
            self.__plat = plat
        else:
            raise Exception(
                "The minimum salary is 35,000 CZK."
            )

    def zobraz_info(self):
        print(f"Name: {self.jmeno}, Age: {self._vek}")

    def _vypocitej_dan(self):
        return self.__plat * 0.20
```
    
</details>

<br>

### Overriding a keyword

---

If a **keyword** collides with a variable name, the *interpreter* will raise a **syntax error**:

In [ ]:
class Zamestnanec:

    def __init__(self, jmeno: str, email: str):
        self.jmeno = jmeno
        self.email = email

In [ ]:
class = Employee("Matous", "matous@gmail.com")  # rezervovaný výraz 'class'

<br>

To avoid this, you can add an underscore as a suffix to the keyword:

In [ ]:
class_ = Zamestnanec("Matous", "matous@gmail.com")  # str_, print_

In [ ]:
print(
    class_.jmeno,
    class_.email,
    sep="\n"
)

<br>

### Magic methods

---

Magic methods are **special methods** in Python (~*double-underscore methods* = **dunder** methods).

Among magic methods is also `__init__`, the *instance constructor*.

<br>

These methods are like the gears that drive Python under the hood:

In [ ]:
len((0.1, 0.2, 0.3))

In [ ]:
(0.1, 0.2, 0.3).__len__()  # len()

In [ ]:
cislo = 3

In [ ]:
cislo.__eq__(3)  # ==

In [ ]:
cislo.__eq__(4)  # ==

In [ ]:
cislo.__add__(7) # +

<br>

The functionality of many objects depends on these methods behind the scenes.

In [ ]:
class Zamestnanec:

    def __init__(self, jmeno: str, email: str):
        self.jmeno = jmeno
        self.email = email

In [ ]:
matous = Zamestnanec("Matous", "matous@gmail.com")

In [ ]:
print(
    matous,
    str(matous),
    repr(matous),
    sep="\n"
)

<br>

If you need to customise the **informational output** for an instance of the `Zamestnanec` class, you can override the `__str__` method:

In [ ]:
class Zamestnanec:

    def __init__(self, jmeno: str, email: str):
        self.jmeno = jmeno
        self.email = email
        
    def __str__(self) -> str:
        return f"Jméno zaměstnance: {self.jmeno}"
    
    # def __repr__(self):
    #     pass

In [ ]:
matous = Zamestnanec("Matous", "matous@gmail.com")

In [ ]:
print(
    matous,
    str(matous),
    repr(matous),
    sep="\n"
)

<br>

You can represent an object in two main ways:
1. overriding the `__str__` method (informal, mainly for end users),
2. overriding the `__repr__` method (formal, for logs and debugging).

In [ ]:
import datetime

In [ ]:
aktualni_datum_cas = datetime.datetime.now()

In [ ]:
aktualni_datum_cas        # __repr__

In [ ]:
repr(aktualni_datum_cas)        # __repr__

In [ ]:
print(aktualni_datum_cas) # __str__

In [ ]:
str(aktualni_datum_cas)   # __str__

In [ ]:
type(aktualni_datum_cas)

<br>

If you want to apply your own methods in your class:

In [ ]:
class Zamestnanec:

    def __init__(self, jmeno: str, email: str):
        self.jmeno = jmeno
        self.email = email
        
    def __str__(self) -> str:
        return f"Employee name: {self.jmeno}"
    
    def __repr__(self):
        return f"{type(self).__name__}(jmeno={self.jmeno!r})"

In [ ]:
matous = Zamestnanec("Matous", "matous@gmail.com")

In [ ]:
print(
    str(matous),
    repr(matous),
    sep="\n"
)

In [ ]:
matous.__dict__

<br>

Using **magic methods** is one of the more advanced features of OOP.

#### Další přetěžování magických metod

---

Adding integers and floats is well known:

In [ ]:
print(11 + 9)

<br>

Similarly, you can imagine *concatenating* strings:

In [ ]:
print('Matouš' + ' Holinka')

<br>

But what if you need to **add two-dimensional vectors**:

In [ ]:
class Vektor2D:
    def __init__(self, rozmer_x: int, rozmer_y: int):
        self.rozmer_x = rozmer_x
        self.rozmer_y = rozmer_y

In [ ]:
v1 = Vektor2D(rozmer_x=5, rozmer_y=4)
v2 = Vektor2D(rozmer_x=7, rozmer_y=6)

In [ ]:
print(v1 + v2)

<br>

In this case, addition is not possible.

For the `Vektor2D` object, addition is not supported yet.

In [ ]:
class Vektor2D:
    def __init__(self, rozmer_x: int, rozmer_y: int):
        self.rozmer_x = rozmer_x
        self.rozmer_y = rozmer_y
        
    def __add__(self, druhy_vektor: Vektor2D):  # +
        return [
            self.rozmer_x + druhy_vektor.rozmer_x,  # Vektor2D(x) + Vektor2D(x) = 5 + 7
            self.rozmer_y + druhy_vektor.rozmer_y   # Vektor2D(y) + Vektor2D(y) = 4 + 6
        ]

In [ ]:
v1 = Vektor2D(rozmer_x=5, rozmer_y=4)
v2 = Vektor2D(rozmer_x=7, rozmer_y=6)

In [ ]:
print(v1 + v2)

<br>

### 🧠 EXERCISE 🧠, Create a class `NakupniKosik` with the following requirements:
---

1. Definuj| třídu `NakupniKosik`,
2. nachystej instanční konstruktor, který potřebuje **nemá parametry**,
3. uprav instanční konstruktor, aby nachystal *weak private* atribut `polozky` jako `dict`,
4. definuj magickou metodu `__str__`, která vrací `"Nákupní košík: <POLOZKY_NAKUPNIHO_KOSIKU>"`,
5. vytvoř instanční metodu `pridat_polozku` s parametry `nazev: str` a `mnozstvi: int`,
6. instanční metoda `pridat_polozku` nejprve ověří parametr `nazev` pomocí další statické metody `overit_polozku`,
7. pokud `nazev` ve `overit_polozku` statické metodě potvrdí, že je argument `str` a současně není prázdný, vrací `True`, jinak `False`,
8. pokud statická metoda `overit_polozku` vrací `True`, tak instanční metoda `pridat_polozku`, založí hodnotu v `polozky` jako `polozky[nazev] = mnozstvi`,
9. pokud už daný produkt máš v `polozky`, potom **inkrementuj** původní hodnotu a novou hodnotu,
10. vytvoř instanční metodu `odebrat_polozku` s parametry `nazev: str`,
11. pokud je argument v `nazev` součástí `polozky`, odstraň jej,
12. vytvoř třídní metodu `vytvorit_s_ukazkovymi_polozkami`,
13. metoda `vytvorit_s_ukazkovymi_polozkami` do původního prázdného `dict` se jménem `polozky` sama přidá **5 jablek** a **3 hrušky**.

In [ ]:
kosik = NakupniKosik.vytvorit_s_ukazkovymi_polozkami()

In [ ]:
print(kosik)

In [ ]:
kosik.pridat_polozku("banány", 2)

In [ ]:
print(kosik)

In [ ]:
kosik.odebrat_polozku("hrušky")

In [ ]:
print(kosik)

<details>
    <summary>▶️ Solution</summary>
    
```python
class NakupniKosik:
    def __init__(self):
        self._polozky = {}

    def __str__(self):
        return f"Nákupní košík: {self._polozky}"

    def pridat_polozku(self, nazev: str, mnozstvi: int) -> None:
        if self.overit_polozku(nazev):
            self._polozky[nazev] = self._polozky.get(nazev, 0) + mnozstvi

    def odebrat_polozku(self, nazev: str):
        if nazev in self._polozky:
            del self._polozky[nazev]

    @staticmethod
    def overit_polozku(nazev: str):
        return isinstance(nazev, str) and nazev != ""

    @classmethod
    def vytvorit_s_ukazkovymi_polozkami(cls):
        kosik = cls()
        kosik.pridat_polozku("jablka", 5)
        kosik.pridat_polozku("hrušky", 3)
        return kosik
```
</details>

<br>

### 🧠 EXERCISE 🧠, Create a class `Uzivatel` with the following requirements:
---

- Define a class `Uzivatel`,
- prepare an instance constructor that takes parameters `vlastnik` and `naposledy_aktivni`,
- make the instance attribute `vlastnik` protected,
- create a method `zobraz_uzivatele` that returns the user name,
- create a method `prihlaseni_uzivatele` that creates a **private instance attribute** `nyni_aktivni` to store the current date and time,
- create a method `zobraz_prihlaseni` that prints the user's current login time (unformatted),
- create the magic method `__str__` that formats the `str` as: `Uživatel: <vlastnik> je aktivní od: <dd/mm/YYYY HH:MM:SS>.`,
- create a static method `je_aktivni` that takes parameters `posledni_prihlaseni` and `aktualni_prihlaseni`,
- the method `je_aktivni` returns `True`/`False` if the difference between `_nyni_aktivni` and `naposledy_aktivni` is less than 365 days.

In [ ]:
from pandas import to_datetime, Timestamp

<details>
    <summary>▶️ Solution</summary>
    
```python
class Uzivatel:
    """
    Object representing a user account for a
    fictional web project for buying books.
    """

    def __init__(self, vlastnik: str):
        self.__vlastnik = vlastnik
        self.naposledy_aktivni = Timestamp(2021, 4, 5, 11, 11, 11)

    def prihlaseni_uzivatele(self):
        self._nyni_aktivni = to_datetime('today')

    def zobraz_prihlaseni(self):
        return self._nyni_aktivni

    def zobraz_uzivatele(self):
        return self.__vlastnik

    def __str__(self) -> str:
        return f"Uživatel: {self.zobraz_uzivatele()} je aktivní od: {self.zobraz_prihlaseni().strftime('%d/%m/%Y %H:%M:%S')}."

    @staticmethod
    def je_aktivni(posledni_prihlaseni, aktualni_prihlaseni: str) -> bool:
        """
        Return True if the user last logged in less than 365 days ago,
        otherwise return False.
        """
        return (Timestamp(aktualni_prihlaseni) - Timestamp(posledni_prihlaseni)).days < 365
```
</details>

[Feedback form after the second lesson](https://forms.gle/Y2bKaUkHPtAK299s5)

---